# ⚖️ Egyptian Legal RAG Assistant (المساعد القانوني المصري)
Notebook محسن لمعالجة ملفات PDF، التقسيم المبني على المواد القانونية، واستخدام تضمينات `BAAI/bge-m3` مع `ChromaDB` و `Ollama`.

In [ ]:
# ============================================
# الخلية 1: الاستيرادات وإعداد البيئة
# ============================================
import os
import sys
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import ollama
from pypdf import PdfReader

# إعداد المسارات و D: drive cache لتوفير مساحة C:
D_CACHE = Path("D:/hf_cache")
if D_CACHE.parent.exists():
    D_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(D_CACHE)
    os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(D_CACHE)

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw_documents"
VECTOR_DIR = DATA_DIR / "vector_store"

RAW_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("✅ تم استيراد المكتبات وتجهيز البيئة بنجاح")
print(f"📁 مجلد المستندات: {RAW_DIR.resolve()}")
print(f"📁 مجلد التخزين: {VECTOR_DIR.resolve()}")

In [ ]:
# ============================================
# الخلية 2: دالة تنظيف النص العربي والمعايرة (NFKC)
# ============================================
def clean_and_normalize_arabic(text: str) -> str:
    """
    معالجة التشويش ونصوص PDF العربية:
    1. تحويل الحروف المدمجة Presentation Forms إلى حروف عربية قياسية عبر NFKC
    2. إزالة التشكيل
    3. إزالة رموز التحكم غير القابلة للطباعة مع الحفاظ على الأرقام والترقيم القانوني
    """
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_pdf_text(file_path: Path) -> str:
    reader = PdfReader(str(file_path))
    pages_text = []
    for page in reader.pages:
        t = page.extract_text() or ""
        if t.strip():
            pages_text.append(t)
    return clean_and_normalize_arabic("\n".join(pages_text))

pdf_files = list(RAW_DIR.glob("*.pdf"))
print(f"📄 تم العثور على {len(pdf_files)} ملفات PDF:")
for f in pdf_files:
    text = extract_pdf_text(f)
    print(f"   - {f.name}: {len(text)} حرف")

In [ ]:
# ============================================
# الخلية 3: التقسيم حسب المواد القانونية (Article-Based Chunking)
# ============================================
def extract_article_number(text: str) -> str:
    patterns = [
        r"(?:المادة|مادة)\s*[\(\（]?\s*([0-9\u0660-\u0669]+|الأولى|الثانية|الثالثة|الرابعة|الخامسة|السادسة|السابعة|الثامنة|التاسعة|العاشرة|[أ-ي\s\/]+)\s*[\)\）]?",
        r"[\(\（]\s*(?:المادة|مادة)\s*([0-9\u0660-\u0669]+|الأولى|الثانية|الثالثة|الرابعة|الخامسة|السادسة|السابعة|الثامنة|التاسعة|العاشرة|[أ-ي\s\/]+)\s*[\)\）]"
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return re.sub(r"^[\(\（\:\-\.\s]+|[\)\）\:\-\.\s]+$", "", match.group(1).strip())
    return "عام"

def parse_law_articles(text: str, law_title: str):
    article_regex = re.compile(
        r"(?=(?:^|\n)\s*[\(\（\s]*(?:المادة|مادة)\s*[\(\（\s]*"
        r"(?:\d+|[\u0660-\u0669]+|الأولى|الثانية|الثالثة|الرابعة|الخامسة|السادسة|السابعة|الثامنة|التاسعة|العاشرة|[أ-ي\s\/]+)"
        r"\s*[\)\）\s]*[\)\）\s]*[\:\-\.]?)",
        re.MULTILINE | re.IGNORECASE
    )
    blocks = article_regex.split(text)
    chunks = []
    count = 0
    for block in blocks:
        b = block.strip()
        if not b or len(b) < 15:
            continue
        art_num = extract_article_number(b)
        words = b.split()
        if len(words) > 800:
            sub_size = 500
            overlap = 60
            for i in range(0, len(words), sub_size - overlap):
                count += 1
                chunks.append({
                    "chunk_id": f"{law_title}_art_{art_num}_p{count}",
                    "law_title": law_title,
                    "article_number": art_num,
                    "content": " ".join(words[i:i+sub_size])
                })
        else:
            count += 1
            chunks.append({
                "chunk_id": f"{law_title}_art_{art_num}_{count}",
                "law_title": law_title,
                "article_number": art_num,
                "content": b
            })
    return chunks

all_chunks = []
for pdf in pdf_files:
    law_name = pdf.stem.replace("-", " ")
    raw_txt = extract_pdf_text(pdf)
    doc_chunks = parse_law_articles(raw_txt, law_name)
    all_chunks.extend(doc_chunks)
    print(f"📄 {pdf.name}: {len(doc_chunks)} قطعة (مادة قانونية)")

print(f"\n📝 إجمالي القطع المستخرجة بالكامل: {len(all_chunks)}")

In [ ]:
# ============================================
# الخلية 4: توليد التضمينات بنموذج BAAI/bge-m3
# ============================================
model_name = "BAAI/bge-m3"
print(f"⏳ تحميل نموذج التضمين: {model_name}...")
embedding_model = SentenceTransformer(model_name)

texts = [c["content"] for c in all_chunks]
print(f"⏳ توليد تضمينات لـ {len(texts)} قطعة...")
embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=16)

print(f"✅ تم توليد {len(embeddings)} تضمين بنجاح! الأبعاد: {embeddings[0].shape}")

In [ ]:
# ============================================
# الخلية 5: الحفظ في قاعدة البيانات المتجهية (ChromaDB)
# ============================================
client = chromadb.PersistentClient(path=str(VECTOR_DIR))

try:
    client.delete_collection("egyptian_law")
except Exception:
    pass

collection = client.create_collection(
    name="egyptian_law",
    metadata={"hnsw:space": "cosine"}
)

metadatas = [{"law_title": c["law_title"], "article_number": c["article_number"]} for c in all_chunks]
ids = [c["chunk_id"] for c in all_chunks]

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ تم حفظ {collection.count()} قطعة قانونية في ChromaDB بنجاح!")

In [ ]:
# ============================================
# الخلية 6: دوال الاسترجاع وتوليد الإجابة (Retrieve & Generate)
# ============================================
def retrieve(query: str, k: int = 5):
    q_vec = embedding_model.encode([query])[0].tolist()
    return collection.query(query_embeddings=[q_vec], n_results=k)

def generate_answer(question: str, context_blocks: list, model_name: str = "qwen2.5:7b"):
    context_text = "\n\n".join(context_blocks)
    prompt = f"""أنت مساعد قانوني مصري خبير ومباشر ودقيق جداً.
مهمتك: اقرأ النصوص المرجعية المرفقة من التشريعات المصرية وأجب عن السؤال الموجه بدقة متناهية.

قواعد صارمة للإجابة:
1. استند فقط وحصرياً إلى المعلومات الواردة في النصوص المرجعية المرفقة.
2. اذكر رقم المادة واسم القانون بالحرف كما هو مكتوب في النصوص المرفقة.
3. لا تقم بتخمين أو إضافة أي معلومات قانونية من خارج النص.
4. إذا لم تجد الإجابة في النصوص المرفقة، اكتب فقط: "لا توجد معلومات كافية في المستندات المتاحة للإجابة على هذا السؤال."

### النصوص المرجعية:
{context_text}

### السؤال:
{question}

### الإجابة:"""
    
    try:
        resp = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], options={"temperature": 0.05})
        return resp["message"]["content"]
    except Exception:
        resp = ollama.chat(model="qwen2.5:3b", messages=[{"role": "user", "content": prompt}], options={"temperature": 0.05})
        return resp["message"]["content"]

def query_rag(question: str, k: int = 5):
    res = retrieve(question, k)
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    
    context_blocks = []
    sources = []
    art_nums = []
    for i, doc in enumerate(docs):
        title = metas[i].get("law_title", "")
        num = metas[i].get("article_number", "")
        sources.append(f"{title} (مادة {num})")
        if num != "عام": art_nums.append(num)
        context_blocks.append(f"[مصدر {i+1} - {title} - مادة {num}]:\n{doc}")
        
    answer = generate_answer(question, context_blocks)
    return {"answer": answer, "sources": list(set(sources)), "article_numbers": list(set(art_nums))}

print("✅ تم إعداد نظام RAG المطور بنجاح!")

In [ ]:
# ============================================
# الخلية 7: اختبار سؤال واحد
# ============================================
q = "ما هي اللغة الرسمية للدولة؟"
res = query_rag(q)
print(f"❓ السؤال: {q}")
print(f"📝 الإجابة:\n{res['answer']}")
print(f"📚 المصادر: {res['sources']}")

In [ ]:
# ============================================
# الخلية 8: تقييم شامل (10 أسئلة قانونية)
# ============================================
test_queries = [
    "ما هي عقوبة السير عكس الاتجاه؟",
    "ما هي اللغة الرسمية للدولة؟",
    "كم تبلغ غرامة مخالفة الإشارات؟",
    "كم إجازة العامل السنوية؟",
    "ما هو المصدر الرئيسي للتشريع؟",
    "هل يجوز السير عكس الاتجاه؟",
    "ما هي السرعة القصوى داخل المدن؟",
    "ما هي مدة عقد العمل؟",
    "كم غرامة التوقف في الأماكن الممنوعة؟",
    "من هو مصدر السلطات في مصر؟"
]

eval_rows = []
for q in test_queries:
    try:
        res = query_rag(q)
        ans = res["answer"]
        is_grounded = "لا توجد معلومات" not in ans and len(ans) > 15
        eval_rows.append({
            "السؤال": q,
            "المصادر": ", ".join(res["sources"][:2]),
            "الإجابة": ans[:120] + "..." if len(ans) > 120 else ans,
            "الحالة": "✅" if is_grounded else "⚠️"
        })
        print(f"✅ تمت معالجة: {q}")
    except Exception as e:
        print(f"❌ خطأ في {q}: {e}")

df_eval = pd.DataFrame(eval_rows)
df_eval.to_csv("../data/evaluation_results.csv", index=False, encoding="utf-8-sig")
print("\n📊 جدول نتائج التقييم الشامل:")
df_eval